# INITIAL IMPORT

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

In [2]:
from src.config import Configuration
from src.tetris import TetrisConfiguration

T_CONFIG = TetrisConfiguration(
    board_w=10,
    board_h=20,
)

CONFIG = Configuration(
    max_board_size_w=10,
    max_board_size_h=20,
)

# Game

In [3]:
from src.tetris import Board, PieceEnum, Queue, ActionEnum, ActivePiece, Tetris, RotationEnum, ROTATION_DIR

In [4]:
game = Tetris()
print('=== SPAWNED from queue ===')
game.print_state()

print('=== LEFT ===')
game.move_active_piece(ActionEnum.LEFT)
game.print_state()

print('=== RIGHT x2 ===')
game.move_active_piece(ActionEnum.RIGHT)
game.move_active_piece(ActionEnum.RIGHT)
game.print_state()

print('=== ROTATE CW ===')
game.move_active_piece(ActionEnum.ROTATE_CW)
game.print_state()

print('=== ROTATE 180 ===')
game.move_active_piece(ActionEnum.ROTATE_180)
game.print_state()

print('=== ROTATE CCW ===')
game.move_active_piece(ActionEnum.ROTATE_CCW)
game.print_state()

print('=== HOLD (swapped with hold slot) ===')
game.move_active_piece(ActionEnum.HOLD)
game.print_state()
print(f'piece={game.active_piece.type} can_hold={game.can_hold}\n')

print('=== DROP + LOCK + SPAWN next ===')
game.move_active_piece(ActionEnum.DROP)
game.print_state()
print(f'piece={game.active_piece.type} pos=({game.active_piece.x},{game.active_piece.y})\n')

print('=== HOLD again (swap back) ===')
lines =game.move_active_piece(ActionEnum.HOLD)
game.print_state()
print(f'piece={game.active_piece.type}\n')

print(f'=== DIRECT: hard_drop + lock_piece (cleared {lines} lines) ===')
lines = game.move_active_piece(ActionEnum.DROP)
game.print_state()
print()

print('=== BOARD: get_ghost_y + hard_drop ===')
gy = game.board.get_ghost_y(game.active_piece)
print(f'ghost Y from y={game.active_piece.y}: {gy}')
drop_dist = game.board.hard_drop(game.active_piece)
print(f'dropped {drop_dist} rows, now at y={game.active_piece.y}')
game.print_state()

=== SPAWNED from queue ===
Current Board:
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
Active Piece: J at (3, 20) with rotation SPAWN
Next Piece: S
Hold Piece: None
Can Hold: True
=== LEFT ===
Current Board:
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
Active Piece: J at (2, 20) with rotation SPAWN
Next Piece: S
Hold Piece: None
Can Hold: True
=== RIGHT x2 ===
Current Board:
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
Active Piece: J at (4, 20) with rotation SPAWN
Next 

# Move Searcher

In [5]:
from src.tetris import MoveSearcher

game = Tetris(
    # playfield='G'
    playfield=''.join([
        'GGNGGGGGGG',
        'GGNGGGGGGG',
        'GNNNGGGGGG',
        'GNNGGGGGGG',
        'GGNGGGGGGG',
        'NNNGGGGGGG',
        'NNGGGGGGGG',
    ]),
    active_piece='T'
)

game.print_state(include_vanish_zone=True)

all_placements = MoveSearcher(game).get_all_placements()
print(f'Found {len(all_placements)} unique placements for piece')
all_placements

Current Board:
..........
....X.....
...XXX....
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..XXXXXXXX
...XXXXXXX
XX.XXXXXXX
X..XXXXXXX
X...XXXXXX
XX.XXXXXXX
XX.XXXXXXX
Active Piece: T at (3, 20) with rotation SPAWN
Next Piece: I
Hold Piece: None
Can Hold: True
Found 39 unique placements for piece


[{'state': (3, 6, 0),
  'sequence': [<ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DROP: 5>],
  'bitmap': array([1019, 1019, 1009, 1017, 1019, 1016, 1020,   56,   16,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0], dtype=uint32),
  'lines_cleared': 0},
 {'state': (2, 6, 0),
  'sequence': [<ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionEnum.DOWN: 7>,
   <ActionE

In [6]:
i = 0

In [7]:
game.board.print_board(
    b_board=all_placements[i]['bitmap'])
print(f"Placement {i}: {all_placements[i]['state']} lines_cleared={all_placements[i]['lines_cleared']}")
i+=1

..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
....X.....
...XXX....
..XXXXXXXX
...XXXXXXX
XX.XXXXXXX
X..XXXXXXX
X...XXXXXX
XX.XXXXXXX
XX.XXXXXXX
Placement 0: (3, 6, 0) lines_cleared=0


### See sequence

In [8]:
i = 37

In [9]:
i+=1

In [10]:
j = 0
game_aux = Tetris(playfield=''.join([
        'GGNGGGGGGG',
        'GGNGGGGGGG',
        'GNNNGGGGGG',
        'GNNGGGGGGG',
        'GGNGGGGGGG',
        'NNNGGGGGGG',
        'NNGGGGGGGG',
    ]),
    active_piece='T')
seq = all_placements[i]['sequence']
print(f'Action sequence to achieve placement {i}: {seq}')

Action sequence to achieve placement 38: [<ActionEnum.DOWN: 7>, <ActionEnum.DOWN: 7>, <ActionEnum.DOWN: 7>, <ActionEnum.DOWN: 7>, <ActionEnum.DOWN: 7>, <ActionEnum.DOWN: 7>, <ActionEnum.DOWN: 7>, <ActionEnum.DOWN: 7>, <ActionEnum.DOWN: 7>, <ActionEnum.DOWN: 7>, <ActionEnum.DOWN: 7>, <ActionEnum.DOWN: 7>, <ActionEnum.DOWN: 7>, <ActionEnum.DOWN: 7>, <ActionEnum.LEFT: 0>, <ActionEnum.LEFT: 0>, <ActionEnum.ROTATE_CW: 2>, <ActionEnum.LEFT: 0>, <ActionEnum.DOWN: 7>, <ActionEnum.ROTATE_CCW: 3>, <ActionEnum.ROTATE_CCW: 3>, <ActionEnum.DOWN: 7>, <ActionEnum.ROTATE_180: 4>, <ActionEnum.DROP: 5>]


In [11]:

action = seq[j]
print(f'Action: {action}')
game_aux.move_active_piece(action)
game_aux.print_state(include_vanish_zone=True)
j+=1

Action: ActionEnum.DOWN
Current Board:
..........
..........
....X.....
...XXX....
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..XXXXXXXX
...XXXXXXX
XX.XXXXXXX
X..XXXXXXX
X...XXXXXX
XX.XXXXXXX
XX.XXXXXXX
Active Piece: T at (3, 19) with rotation SPAWN
Next Piece: S
Hold Piece: None
Can Hold: True


# With env

In [12]:
from src.models import TetrisEnv

env = TetrisEnv(CONFIG, T_CONFIG)

state = env.reset()[0]
print(state.keys())
state

dict_keys(['boards', 'queues', 'queue_idx', 'placement_mask'])


{'boards': array([[[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        ...,
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0.,

In [13]:
i = -1

In [14]:
i+=1
state['boards'][i]


array([[0., 0., 0., 0., 1., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 

In [15]:
import numpy as np

grid = state['boards'][i]
# Convert float32 occupancy grid to bitmap (col 0 = LSB)
row_ints = np.zeros(grid.shape[0], dtype=np.uint32)
for x in range(grid.shape[1]):
    row_ints |= (grid[:, x] > 0.5).astype(np.uint32) << x

game.board.color_map = False
game.board.print_board(
    b_board=row_ints,
    include_vanish_zone=True
)
i+=1

..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
....XX....
....XX....


In [16]:
queues = state['queues']
print(queues.shape)
print(queues) 

print()
queues_idx = state['queue_idx']
print(queues_idx.shape)
print(queues_idx) 

(2, 7, 8)
[[[0. 0. 1. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 1.]
  [0. 0. 0. 0. 0. 0. 0. 1.]
  [0. 0. 0. 0. 0. 0. 1. 0.]
  [0. 0. 0. 1. 0. 0. 0. 0.]
  [0. 0. 0. 0. 1. 0. 0. 0.]
  [0. 1. 0. 0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0. 0. 0. 1.]
  [0. 0. 1. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 1. 0.]
  [0. 0. 0. 1. 0. 0. 0. 0.]
  [0. 0. 0. 0. 1. 0. 0. 0.]
  [0. 1. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 1. 0. 0.]]]

(128,)
[0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [17]:
masks = state['placement_mask']
print(masks.shape)
print(masks) 


(128,)
[ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False]


# Heuristics

In [54]:
game = Tetris(
    # playfield='G',
    # playfield=''.join([
    #     'GGNGGGGGGG',
    #     'GGNGGGGGGG',
    #     'GNNNGGGGGG',
    #     'GNNGGGGGGG',
    #     'GGNGGGGGGG',
    #     'NNNGGGGGGG',
    #     'NNGGGGGGGG',
    # ]),
    playfield=''.join([
        'NGGGGGGGGG',
        'NGGGGGGGGG',
        'NGGGGGGGGG',
        'NGGGGGGGGG',
        'NGGGGGGGGG',
        # 'GNGGGGGGGG',
    ]),

    active_piece='T'
)
game.print_state()

Current Board:
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
.XXXXXXXXX
.XXXXXXXXX
.XXXXXXXXX
.XXXXXXXXX
.XXXXXXXXX
Active Piece: T at (3, 20) with rotation SPAWN
Next Piece: T
Hold Piece: None
Can Hold: True


In [55]:
pieces = [
    9, 9, 7, 8, 9, 7, 8, 
]

w = 0
for i, p in enumerate(pieces, start=1):
    w += p * i
w

223

In [56]:
from src.tetris import HeuristicEvaluator

evaluator = HeuristicEvaluator()

eval_result = evaluator.evaluate(game.board)
print(eval_result)
print(eval_result.compute_total())

HeuristicsResult(
  lines_cleared=0, 
  tetrises=0, 
  blocks=45, 
  weighted_blocks=135, 
  clearable_lines=4, 
  roughness=5, 
  col_holes=0, 
  connected_holes=0, 
  blocks_above_holes=0, 
  pit_hole_percent=0.00, 
  deepest_well=0
)
-2.1
